## Step 1: Setup and File Discovery
Import the libraries, ask for the bronze folder path, and list all CSV files found.

In [3]:
import os
import pandas as pd

# Ask the user for the folder path (press Enter to use the default)
default_path = "../data/bronze/csv"
bronze_path = input(f"Enter bronze folder path [{default_path}]: ").strip()
if bronze_path == "":
    bronze_path = default_path

all_files = os.listdir(bronze_path)
csv_files = []

# Keep only the CSV files
i = 0
while i < len(all_files):
    if all_files[i].lower().endswith(".csv"):
        csv_files.append(all_files[i])
    i += 1

csv_files.sort()
print(f"Found {len(csv_files)} CSV files:")

i = 0
while i < len(csv_files):
    print(f"{i + 1}. {csv_files[i]}")
    i += 1

Found 20 CSV files:
1. amsterdam_weekdays.csv
2. amsterdam_weekends.csv
3. athens_weekdays.csv
4. athens_weekends.csv
5. barcelona_weekdays.csv
6. barcelona_weekends.csv
7. berlin_weekdays.csv
8. berlin_weekends.csv
9. budapest_weekdays.csv
10. budapest_weekends.csv
11. lisbon_weekdays.csv
12. lisbon_weekends.csv
13. london_weekdays.csv
14. london_weekends.csv
15. paris_weekdays.csv
16. paris_weekends.csv
17. rome_weekdays.csv
18. rome_weekends.csv
19. vienna_weekdays.csv
20. vienna_weekends.csv


## Step 2: File-Level Summary
For each file, record the number of rows, columns, null values, and duplicated rows.
This gives a first overview of data quality before any cleaning.

In [4]:
summary_rows = []
columns_by_file = {}

i = 0
while i < len(csv_files):
    file_name = csv_files[i]
    df = pd.read_csv(os.path.join(bronze_path, file_name))

    # Store the column names to compare schemas later
    columns_by_file[file_name] = list(df.columns)

    summary_rows.append({
        "file": file_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "total_nulls": int(df.isnull().sum().sum()),
        "duplicates": int(df.duplicated().sum()),
    })
    i += 1

summary_df = pd.DataFrame(summary_rows)
print("Total rows across all files:", summary_df["rows"].sum())
summary_df

Total rows across all files: 51707


,file,rows,columns,total_nulls,duplicates
0,amsterdam_weekdays.csv,1103,20,0,0
1,amsterdam_weekends.csv,977,20,0,0
2,athens_weekdays.csv,2653,20,0,0
3,athens_weekends.csv,2627,20,0,0
4,barcelona_weekdays.csv,1555,20,0,0
5,barcelona_weekends.csv,1278,20,0,0
6,berlin_weekdays.csv,1284,20,0,0
7,berlin_weekends.csv,1200,20,0,0
8,budapest_weekdays.csv,2074,20,0,0
9,budapest_weekends.csv,1948,20,0,0


## Step 3: Schema Consistency Check
Compare the column names of every file against the first file.
If all files share the same schema, they can be appended safely in the Silver layer.

In [5]:
base_file = csv_files[0]
base_cols = columns_by_file[base_file]

i = 1
while i < len(csv_files):
    current_cols = columns_by_file[csv_files[i]]
    if current_cols == base_cols:
        print(f"{csv_files[i]}: same columns")
    else:
        print(f"{csv_files[i]}: DIFFERENT")
        print("  only in this file:", set(current_cols) - set(base_cols))
        print("  missing from this file:", set(base_cols) - set(current_cols))
    i += 1

amsterdam_weekends.csv: same columns
athens_weekdays.csv: same columns
athens_weekends.csv: same columns
barcelona_weekdays.csv: same columns
barcelona_weekends.csv: same columns
berlin_weekdays.csv: same columns
berlin_weekends.csv: same columns
budapest_weekdays.csv: same columns
budapest_weekends.csv: same columns
lisbon_weekdays.csv: same columns
lisbon_weekends.csv: same columns
london_weekdays.csv: same columns
london_weekends.csv: same columns
paris_weekdays.csv: same columns
paris_weekends.csv: same columns
rome_weekdays.csv: same columns
rome_weekends.csv: same columns
vienna_weekdays.csv: same columns
vienna_weekends.csv: same columns


## Step 4: Sample Inspection
Look at one file in detail: first rows, data types, and basic statistics.

In [6]:
sample = pd.read_csv(os.path.join(bronze_path, csv_files[0]))

display(sample.head())
sample.info()
display(sample.describe())

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
0,0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772
1,1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432
2,2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103
3,3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,2,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663
4,4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,1,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508


<class 'pandas.DataFrame'>
RangeIndex: 1103 entries, 0 to 1102
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  1103 non-null   int64  
 1   realSum                     1103 non-null   float64
 2   room_type                   1103 non-null   str    
 3   room_shared                 1103 non-null   bool   
 4   room_private                1103 non-null   bool   
 5   person_capacity             1103 non-null   float64
 6   host_is_superhost           1103 non-null   bool   
 7   multi                       1103 non-null   int64  
 8   biz                         1103 non-null   int64  
 9   cleanliness_rating          1103 non-null   float64
 10  guest_satisfaction_overall  1103 non-null   float64
 11  bedrooms                    1103 non-null   int64  
 12  dist                        1103 non-null   float64
 13  metro_dist                  1103 non-null   

,Unnamed: 0,realSum,person_capacity,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
count,1103.000000,1103.000000,1103.000000,1103.00000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000,1103.000000
mean,551.000000,545.020526,2.792384,0.30825,0.115141,9.461469,94.362647,1.282865,2.841621,1.089439,271.009899,14.350154,341.541187,23.799081,4.891158,52.364858
std,318.552978,416.974314,1.044151,0.46198,0.319336,0.798201,6.089691,0.740178,2.123245,0.836546,197.046890,10.433764,236.611077,16.487400,0.038882,0.019467
min,0.000000,128.887118,2.000000,0.00000,0.000000,4.000000,20.000000,0.000000,0.015059,0.036530,40.931415,2.167346,50.877318,3.545205,4.775500,52.291100
25%,275.500000,309.797764,2.000000,0.00000,0.000000,9.000000,92.000000,1.000000,1.302058,0.462983,127.909866,6.772912,163.469245,11.390772,4.871000,52.354580
50%,551.000000,430.248635,2.000000,0.00000,0.000000,10.000000,96.000000,1.000000,2.341366,0.856010,208.180311,11.023286,260.257028,18.135084,4.890010,52.365590
75%,826.500000,657.324303,4.000000,1.00000,0.000000,10.000000,98.000000,2.000000,3.648138,1.510629,386.442241,20.462373,469.290623,32.700846,4.907315,52.375260
max,1102.000000,7782.907225,6.000000,1.00000,1.000000,10.000000,100.000000,5.000000,11.187100,4.411915,1888.550428,100.000000,1435.102401,100.000000,5.010770,52.423480


## Step 5: Hidden Missing Values Check (Zeros)
`isnull()` found no nulls, but zeros can still hide missing data.
Count zeros per column across all files, then decide which zeros are valid
(binary flags, studios) and which are suspicious (price, distance, indices).

In [7]:
zero_counts = {}
total_rows = 0

i = 0
while i < len(csv_files):
    df = pd.read_csv(os.path.join(bronze_path, csv_files[i]))
    total_rows += len(df)

    cols = list(df.columns)
    j = 0
    while j < len(cols):
        col = cols[j]
        # Skip the leftover index column
        if col != "Unnamed: 0":
            # For boolean columns, False counts as zero
            zero_counts[col] = zero_counts.get(col, 0) + int((df[col] == 0).sum())
        j += 1
    i += 1

zero_df = pd.DataFrame({
    "column": list(zero_counts.keys()),
    "zero_count": list(zero_counts.values()),
})
zero_df["zero_pct"] = (zero_df["zero_count"] / total_rows * 100).round(2)
zero_df = zero_df.sort_values("zero_pct", ascending=False).reset_index(drop=True)
zero_df

,column,zero_count,zero_pct
0,room_shared,51341,99.29
1,host_is_superhost,38475,74.41
2,multi,36642,70.86
3,biz,33599,64.98
4,room_private,33014,63.85
5,bedrooms,4485,8.67
6,realSum,0,0.00
7,room_type,0,0.00
8,person_capacity,0,0.00
9,cleanliness_rating,0,0.00


## Step 6: Bedrooms = 0 Check
Verify that zero bedrooms correspond to studios (entire place) and not to bad data.
The combined data is used for inspection only and is not saved.

In [8]:
# Read every file into a list, then combine in memory (inspection only)
frames = []
i = 0
while i < len(csv_files):
    frames.append(pd.read_csv(os.path.join(bronze_path, csv_files[i])))
    i += 1

all_data = pd.concat(frames, ignore_index=True)

# Which room types have zero bedrooms?
display(pd.crosstab(all_data["room_type"], all_data["bedrooms"]))

# Minimum values of the numeric columns that should never be zero
check_cols = ["realSum", "person_capacity", "dist", "metro_dist",
              "attr_index", "rest_index"]
display(all_data[check_cols].min())

bedrooms,0,1,2,3,4,5,6,8,9,10
room_type,,,,,,,,,,
Entire home/apt,4026,18489,8650,1410,67,4,2,0,0,0
Private room,459,17478,640,67,29,6,0,2,10,2
Shared room,0,366,0,0,0,0,0,0,0,0


realSum            34.779339
person_capacity     2.000000
dist                0.015045
metro_dist          0.002301
attr_index         15.152201
rest_index         19.576924
dtype: float64

## Observations
- File naming pattern: {city}_{weekdays|weekends}.csv (10 cities x 2 = 20 files)
- Same schema across files: yes (20 columns)
- Unnamed index column present: yes (Unnamed: 0), to be dropped in Silver
- Listing ID column present: no
- Nulls and duplicates: none found (0 nulls, 0 duplicates in every file)
- Zeros: valid (binary flags, studios); no hidden missing values
- Total rows: 51,707